# LLM Annotators via aiXplain (Claude + ChatGPT + optional Gemini)

This Colab notebook uses the **aiXplain** API (https://aixplain.com) to call **Claude**,
**ChatGPT**, and optionally **Gemini** as **independent annotators** for the two
second-evaluator tasks of the inter-annotator agreement (IAA) survey:

* **Task 2 - Tier E validation** - does each sampled Tier E paper contain reliability
  evidence, and at which tier (B / C / D)?
* **Task 3 - Compliance audit** - score each recent paper on the 13-item checklist.

Each model codes every paper **independently**. The notebook reports pairwise Cohen's
kappa between every pair of models, Fleiss' kappa across all of them when there are three
or more, a **majority-vote consensus**, and the per-item rates. It checkpoints after every
paper, resumes where it stopped, and re-calls only the model that still needs a paper.

You run with **two** models (leave the Gemini ID blank) or **three** (paste a Gemini ID).

## Important - honest disclosure

These annotators are **large language models, not humans.** If you use these codings or
the kappa in the paper, the manuscript **must say so**: describe the annotators as LLMs,
name the models and versions, and give the run date and the prompt. LLMs share training
data and failure modes, so their agreement runs optimistic; a third model does not fix
that, it only adds a majority-vote consensus and a proper multi-rater kappa.

## Step 1 - Install the aiXplain SDK

In [ ]:
!pip -q install aiXplain pdfplumber requests pandas scikit-learn statsmodels

## Step 2 - API key

Get a Team API Key from your aiXplain account (Dashboard -> API Keys). The SDK reads the
key at import time, so this cell sets it first. **aiXplain charges credits per call.**

In [ ]:
import os, getpass

AIXPLAIN_API_KEY = ""  # paste your aiXplain Team API Key, or leave blank to be prompted
if not AIXPLAIN_API_KEY:
    AIXPLAIN_API_KEY = getpass.getpass("aiXplain Team API Key: ")

os.environ["TEAM_API_KEY"] = AIXPLAIN_API_KEY
os.environ["AIXPLAIN_API_KEY"] = AIXPLAIN_API_KEY
os.environ["AIXPLAIN_TEAM_API_KEY"] = AIXPLAIN_API_KEY
print("aiXplain key set in environment.")

## Step 3 - Choose the model IDs

The search cell lists marketplace IDs for Claude, GPT-4o, and Gemini. Paste the IDs you
want into the next cell. Claude and GPT are required; **Gemini is optional** - leave its
ID blank to run with two models. If a Gemini model returns truncated output (see Step 8),
try a different Gemini ID from the search results.

In [ ]:
from aixplain.factories import ModelFactory

for query in ["Claude", "GPT-4o", "Gemini"]:
    print("=== marketplace search:", query, "===")
    try:
        results = ModelFactory.list(query=query)["results"]
    except Exception as e:
        print("  search failed:", e)
        results = []
    for m in results[:12]:
        print("  ", getattr(m, "id", "?"), "|", getattr(m, "name", "?"))
    print()

In [ ]:
# Paste the IDs you picked from the Step 3 search output.
CLAUDE_MODEL_ID = ""                           # REQUIRED
GPT_MODEL_ID    = "6646261c6eb563165658bbb1"   # GPT-4o (verify against the search)
GEMINI_MODEL_ID = ""                           # OPTIONAL - blank = run with 2 models

import time
_config = [("claude", CLAUDE_MODEL_ID), ("gpt", GPT_MODEL_ID), ("gemini", GEMINI_MODEL_ID)]
MODELS = []
for tag, mid in _config:
    if str(mid).strip():
        MODELS.append((tag, ModelFactory.get(str(mid).strip())))
TAGS = [t for t, _ in MODELS]
assert len(MODELS) >= 2, "Set at least CLAUDE_MODEL_ID and GPT_MODEL_ID."
for tag, model in MODELS:
    print("annotator:", tag, "->", getattr(model, "name", "?"))


class OutOfCredits(Exception):
    pass


def run_model(model, prompt, retries=2):
    """Call an aiXplain model. Returns (text, status); status is
    'ok', 'no_credits', or 'failed'. Requests a large output cap so
    replies are not truncated, and falls back to a plain call if the
    SDK build does not accept the parameters argument. A reply is only
    'ok' if it contains both an opening and a closing brace, so a
    truncated reply is retried rather than accepted."""
    last = ""
    for attempt in range(retries + 1):
        resp, err = None, ""
        try:
            resp = model.run(prompt, parameters={"max_tokens": 1200})
        except Exception as e1:
            err = str(e1)
            try:
                resp = model.run(prompt)
            except Exception as e2:
                err = str(e2)
        if resp is None:
            if "balance" in err.lower() or "ax-bil" in err.lower():
                return err, "no_credits"
            last = "EXCEPTION: " + err
            time.sleep(2.0)
            continue
        data = getattr(resp, "data", None)
        if data is None and isinstance(resp, dict):
            data = resp.get("data") or resp.get("output")
        text = data if isinstance(data, str) else str(data if data is not None else resp)
        low = text.lower()
        if "not_enough_balance" in low or "ax-bil" in low or "billingerror" in low:
            return text, "no_credits"
        if text.count("{") >= 1 and text.count("}") >= 1:
            return text, "ok"
        last = text
        time.sleep(2.0)
    return last, "failed"

## Step 4 - Upload the scaffold CSVs

Upload `task2_tier_e_validation.csv` and `task3_compliance_audit_manual.csv`.

In [ ]:
try:
    from google.colab import files
    print("Upload task2_tier_e_validation.csv and task3_compliance_audit_manual.csv ...")
    files.upload()
except Exception:
    print("Not in Colab - put the two CSV files in the working directory.")

import pandas as pd
df2 = pd.read_csv("task2_tier_e_validation.csv")
df3 = pd.read_csv("task3_compliance_audit_manual.csv")
print("Task 2:", len(df2), "papers   Task 3:", len(df3), "papers")

## Step 5 - Fetch paper text

The real PDF URL is resolved from each paper's ACL Anthology landing page. A paper whose
PDF cannot be retrieved is reported with a status and is skipped, never coded from a stub.

In [ ]:
import requests, io, time, logging
import pdfplumber

logging.getLogger("pdfminer").setLevel(logging.ERROR)
HDRS = {"User-Agent": "Mozilla/5.0 (academic research)"}

def resolve_pdf_url(acl_url):
    base = str(acl_url).rstrip("/")
    try:
        html = requests.get(base + "/", timeout=30, headers=HDRS).text
        key = 'citation_pdf_url" content="'
        j = html.find(key)
        if j != -1:
            j += len(key)
            end = html.find('"', j)
            if end > j:
                return html[j:end]
    except Exception:
        pass
    return base + ".pdf"

def fetch_paper_text(acl_url, max_chars=45000):
    pdf_url = resolve_pdf_url(acl_url)
    try:
        r = requests.get(pdf_url, timeout=45, headers=HDRS)
        r.raise_for_status()
    except Exception as e:
        return "", "no_pdf: " + str(e)
    if r.content[:5] != b"%PDF-":
        return "", "not_a_pdf"
    try:
        parts = []
        with pdfplumber.open(io.BytesIO(r.content)) as pdf:
            for page in pdf.pages:
                parts.append(page.extract_text() or "")
        full = chr(10).join(parts).strip()
    except Exception as e:
        return "", "parse_failed: " + str(e)
    if len(full) < 200:
        return full, "no_text_layer"
    return full[:max_chars], "ok"

for tid in ["https://aclanthology.org/2023.acl-long.59",
            "https://aclanthology.org/L12-1175"]:
    t, s = fetch_paper_text(tid)
    print("self-test", tid.split("/")[-1], "->", s, "| chars:", len(t))

## Step 6 - Annotation rubrics and JSON parsing

In [ ]:
import json

def extract_json(text):
    a = text.find("{")
    b = text.rfind("}")
    if a == -1 or b == -1 or b < a:
        return {}
    try:
        return json.loads(text[a:b + 1])
    except Exception:
        return {}

def parse_items(res, items):
    """Extract C01..C13 as 0/1, tolerating booleans, yes/no, nested
    objects, and key variants."""
    def count_hits(d):
        if not isinstance(d, dict):
            return 0
        ks = [str(k).upper().replace(" ", "") for k in d]
        return sum(1 for it in items if any(k.startswith(it) for k in ks))
    if isinstance(res, dict) and count_hits(res) < 7:
        for v in res.values():
            if count_hits(v) >= 7:
                res = v
                break
    out = {}
    if not isinstance(res, dict):
        return out
    norm = {str(k).upper().replace(" ", ""): v for k, v in res.items()}
    for it in items:
        val = None
        for k, v in norm.items():
            if k.startswith(it):
                val = v
                break
        if val is None:
            continue
        s = str(val).strip().lower()
        if s in ("1", "1.0", "true", "yes", "y"):
            out[it] = 1
        elif s in ("0", "0.0", "false", "no", "n"):
            out[it] = 0
    return out

TASK2_RUBRIC = (
"You are one of several independent LLM annotators in a study of inter-annotator "
"agreement (IAA) in NLP. A regex pipeline assigned this paper to Tier E, meaning no "
"reliability evidence was detected. Read the paper and judge independently whether ANY "
"reliability evidence is in fact present. Reliability evidence is ANY of: a numeric IAA "
"value (e.g. kappa = 0.82); a raw percentage agreement (e.g. annotators agreed in 87% "
"of cases); a qualitative reliability procedure (cross-checking, adjudication, training "
"rounds); or an explicit statement that annotators agreed. has_evidence = YES if any of "
"the above appears anywhere, else NO. evidence_tier (only if YES): B = a numeric "
"agreement value but no coefficient named; C = a raw percentage agreement only; D = a "
"qualitative procedure only. Be conservative: answer YES only on explicit textual "
'evidence. Return ONLY a JSON object: {"has_evidence":"YES or NO",'
'"evidence_tier":"B or C or D or empty","key_quote":"the exact sentence or none found",'
'"confidence":"high or medium or low"}')

TASK3_RUBRIC = (
"You are one of several independent LLM annotators scoring a paper on a 13-item "
"annotation reporting checklist for a study of IAA in NLP. Score each item 1 if the "
"paper satisfies it and 0 otherwise. Be strict: score 1 only on explicit textual "
"evidence. C01 Metric named: names a specific IAA coefficient (Cohen kappa, "
"Krippendorff alpha, ICC, Fleiss kappa). C02 Annotator count: states the exact number "
"of annotators. C03 CI reported: a confidence interval accompanies an IAA value. "
"C04 Raw agreement: reports uncorrected percentage agreement. C05 Adjudication: "
"describes how disagreements were resolved. C06 Label scale: states or clearly implies "
"the scale type. C07 Library or version: cites a specific library or version used for "
"IAA. C08 Per-label breakdown: IAA broken down by class or label. C09 Selection: states "
"how annotators were recruited or selected. C10 Compensation: states how annotators "
"were compensated. C11 Session duration: gives annotation time or session length. "
"C12 Expertise: describes annotator background or expertise. C13 Data released: "
"annotations or labels are publicly released. "
"Return ONLY a flat JSON object, with no other text and no markdown code fences. It "
"must have exactly the keys C01 C02 C03 C04 C05 C06 C07 C08 C09 C10 C11 C12 C13, each "
"value the integer 0 or 1, plus a key confidence. "
'Example of the exact format: {"C01":1,"C02":0,"C03":0,"C04":1,"C05":1,"C06":1,'
'"C07":0,"C08":0,"C09":0,"C10":0,"C11":0,"C12":1,"C13":1,"confidence":"high"}')

def build_prompt(rubric, title, abstract, paper_text):
    nl = chr(10)
    return (rubric + nl + nl + "TITLE: " + str(title) + nl +
            "ABSTRACT: " + str(abstract) + nl + nl +
            "PAPER TEXT:" + nl + str(paper_text))

print("Rubrics and parsers ready.")

## Step 7 - Task 2: Tier E validation with every model

Each model codes each paper independently, checkpointed after every paper. On a re-run a
model is re-called only if it has not already finished that paper.

In [ ]:
import os

out2_cols = []
for t in TAGS:
    out2_cols += [t + "_has_evidence", t + "_tier", t + "_quote"]
out2_cols += ["fetch_status"]

if os.path.exists("task2_two_llm_annotators.csv"):
    out2 = pd.read_csv("task2_two_llm_annotators.csv")
    print("Resuming from checkpoint.")
else:
    out2 = df2.copy()
for c in out2_cols:
    if c not in out2.columns:
        out2[c] = ""

DONE = ("YES", "NO", "NO_FULLTEXT")
for i, row in out2.iterrows():
    if all(str(row.get(t + "_has_evidence", "")).strip() in DONE for t in TAGS):
        continue
    text, status = fetch_paper_text(row["ACL_URL"])
    out2.at[i, "fetch_status"] = status
    if status != "ok":
        for t in TAGS:
            out2.at[i, t + "_has_evidence"] = "NO_FULLTEXT"
        out2.to_csv("task2_two_llm_annotators.csv", index=False)
        print("[" + str(i + 1) + "/" + str(len(out2)) + "] " + str(row["acl_id"]) +
              "  SKIPPED - " + status)
        continue
    abstract = str(row.get("abstract_first_300", ""))
    prompt = build_prompt(TASK2_RUBRIC, row["title"], abstract, text)
    notes = []
    for tag, model in MODELS:
        if str(row.get(tag + "_has_evidence", "")).strip() in DONE:
            notes.append(tag + "=kept")
            continue
        reply, st = run_model(model, prompt)
        time.sleep(1.0)
        if st == "no_credits":
            out2.to_csv("task2_two_llm_annotators.csv", index=False)
            raise OutOfCredits("aiXplain account out of credits. Top up, then re-run "
                               "this cell - it resumes from where it stopped.")
        if st != "ok":
            out2.at[i, tag + "_has_evidence"] = "MODEL_FAIL"
            out2.at[i, tag + "_quote"] = str(reply)[:200]
            notes.append(tag + "=FAIL")
            continue
        res = extract_json(reply)
        if not res:
            out2.at[i, tag + "_has_evidence"] = "PARSE_FAIL"
            out2.at[i, tag + "_quote"] = str(reply)[:200]
            notes.append(tag + "=PARSE_FAIL")
            continue
        out2.at[i, tag + "_has_evidence"] = str(res.get("has_evidence", "")).upper().strip()
        out2.at[i, tag + "_tier"] = res.get("evidence_tier", "")
        out2.at[i, tag + "_quote"] = res.get("key_quote", "")
        notes.append(tag + "=" + str(out2.at[i, tag + "_has_evidence"]))
    out2.to_csv("task2_two_llm_annotators.csv", index=False)
    print("[" + str(i + 1) + "/" + str(len(out2)) + "] " + str(row["acl_id"]) +
          "  " + "  ".join(notes))

print("Saved task2_two_llm_annotators.csv")

## Step 8 - Task 3: Compliance audit with every model

Each model's raw reply is saved to a `<model>_raw` column for debugging. On a re-run a
model is re-called only if it has not already finished a paper - so adding Gemini to a
finished Claude+GPT run costs only the Gemini calls. If a model logs `FAIL` for every
paper, open the CSV and read its `_raw` column: a few-character reply means the model is
returning truncated output and you should try a different model ID for it.

In [ ]:
import os
ITEMS = ["C01", "C02", "C03", "C04", "C05", "C06", "C07",
         "C08", "C09", "C10", "C11", "C12", "C13"]

if os.path.exists("task3_two_llm_annotators.csv"):
    out3 = pd.read_csv("task3_two_llm_annotators.csv")
    print("Resuming from checkpoint.")
else:
    out3 = df3.copy()
out3["fetch_status"] = out3.get("fetch_status", "")
for t in TAGS:
    for it in ITEMS:
        if t + "_" + it not in out3.columns:
            out3[t + "_" + it] = ""
    for col in [t + "_status", t + "_raw"]:
        if col not in out3.columns:
            out3[col] = ""

for i, row in out3.iterrows():
    if all(str(row.get(t + "_status", "")).strip() == "ok" for t in TAGS):
        continue
    text, status = fetch_paper_text(row["ACL_URL"])
    out3.at[i, "fetch_status"] = status
    if status != "ok":
        for t in TAGS:
            out3.at[i, t + "_status"] = "no_fulltext"
        out3.to_csv("task3_two_llm_annotators.csv", index=False)
        print("[" + str(i + 1) + "/" + str(len(out3)) + "] " + str(row["acl_id"]) +
              "  SKIPPED - " + status)
        continue
    abstract = str(row.get("abstract_first_200", ""))
    prompt = build_prompt(TASK3_RUBRIC, row["title"], abstract, text)
    notes = []
    for tag, model in MODELS:
        if str(row.get(tag + "_status", "")).strip() == "ok":
            notes.append(tag + "=kept")
            continue
        reply, st = run_model(model, prompt)
        time.sleep(1.0)
        out3.at[i, tag + "_raw"] = str(reply)[:400]
        if st == "no_credits":
            out3.to_csv("task3_two_llm_annotators.csv", index=False)
            raise OutOfCredits("aiXplain account out of credits. Top up, then re-run.")
        if st != "ok":
            out3.at[i, tag + "_status"] = "model_fail"
            notes.append(tag + "=FAIL")
            continue
        got = parse_items(extract_json(reply), ITEMS)
        for it, v in got.items():
            out3.at[i, tag + "_" + it] = v
        out3.at[i, tag + "_status"] = "ok" if len(got) == 13 else "partial"
        notes.append(tag + "=" + str(len(got)) + "/13")
    out3.to_csv("task3_two_llm_annotators.csv", index=False)
    print("[" + str(i + 1) + "/" + str(len(out3)) + "] " + str(row["acl_id"]) +
          "  " + "  ".join(notes))

for t in TAGS:
    print("  " + t + " fully coded:", int((out3[t + "_status"] == "ok").sum()), "papers")
print("Saved task3_two_llm_annotators.csv")

## Step 9 - Agreement and statistics

Reports per-model results, pairwise Cohen's kappa, Fleiss' kappa across all models with
three or more, and a majority-vote consensus. A model that produced no data is named and
excluded, so it never blanks the models that did work.

In [ ]:
import pandas as pd, itertools
from sklearn.metrics import cohen_kappa_score
try:
    from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters
    HAVE_SM = True
except Exception:
    HAVE_SM = False

ITEMS = ["C01", "C02", "C03", "C04", "C05", "C06", "C07",
         "C08", "C09", "C10", "C11", "C12", "C13"]
ORDER = ["claude", "gpt", "gemini"]

# ---- TASK 2 ----
d2 = pd.read_csv("task2_two_llm_annotators.csv")
present = [t for t in ORDER if t + "_has_evidence" in d2.columns]
live = [t for t in present if int(d2[t + "_has_evidence"].isin(["YES", "NO"]).sum()) > 0]
dead = [t for t in present if t not in live]
print("=== TASK 2: Tier E validation ===")
print("models with data:", ", ".join(live) if live else "none",
      ("| NO usable data from: " + ", ".join(dead) if dead else ""))
for t in live:
    ev = d2[t + "_has_evidence"]
    n = int(ev.isin(["YES", "NO"]).sum())
    y = int((ev == "YES").sum())
    fpr = y / n
    print("  " + t + ": evidence", y, "/", n, "-> FPR", round(100 * fpr, 1),
          "% -> Tier E ceiling", round(183 * (1 - fpr) / 665 * 100, 1), "%")
mask = pd.Series(bool(live), index=d2.index)
for t in live:
    mask = mask & d2[t + "_has_evidence"].isin(["YES", "NO"])
if len(live) >= 2 and int(mask.sum()) > 0:
    vote = {t: d2.loc[mask, t + "_has_evidence"].map({"YES": 1, "NO": 0}) for t in live}
    print("  papers coded by all", len(live), "live models:", int(mask.sum()))
    for a, b in itertools.combinations(live, 2):
        print("    Cohen kappa", a, "vs", b, ":",
              round(cohen_kappa_score(list(vote[a]), list(vote[b])), 3))
    if HAVE_SM and len(live) >= 3:
        rows = list(zip(*[list(vote[t]) for t in live]))
        tbl, _ = aggregate_raters(rows)
        print("    Fleiss kappa (" + ", ".join(live) + "):", round(fleiss_kappa(tbl), 3))
    if len(live) >= 3:
        s = sum(vote[t].reset_index(drop=True) for t in live)
        maj = s * 2 > len(live)
        fpr = int(maj.sum()) / len(maj)
        print("    majority vote: FPR", round(100 * fpr, 1), "% -> Tier E ceiling",
              round(183 * (1 - fpr) / 665 * 100, 1), "%")

# ---- TASK 3 ----
d3 = pd.read_csv("task3_two_llm_annotators.csv")
present3 = [t for t in ORDER if t + "_C01" in d3.columns]
def has_data(t):
    for it in ITEMS:
        if pd.to_numeric(d3[t + "_" + it], errors="coerce").notna().sum() > 0:
            return True
    return False
live3 = [t for t in present3 if has_data(t)]
dead3 = [t for t in present3 if t not in live3]
print()
print("=== TASK 3: Compliance audit ===")
print("models with data:", ", ".join(live3) if live3 else "none",
      ("| NO usable data from: " + ", ".join(dead3) if dead3 else ""))
labels = {"C01": "Metric named", "C02": "Annotator count", "C03": "CI reported",
          "C04": "Raw agreement", "C05": "Adjudication", "C06": "Label scale",
          "C07": "Library/version", "C08": "Per-label breakdown", "C09": "Selection",
          "C10": "Compensation", "C11": "Session duration", "C12": "Expertise",
          "C13": "Data released"}
if live3:
    print("per-item compliance rate per model" +
          ("  (+ majority)" if len(live3) >= 3 else "") + ":")
    maj_rates = []
    for it in ITEMS:
        line = "  " + it + " " + labels[it].ljust(20)
        for t in live3:
            v = pd.to_numeric(d3[t + "_" + it], errors="coerce").dropna()
            line += " " + t[:3] + "=" + (str(round(100 * v.mean(), 1)) if len(v) else "NA")
        if len(live3) >= 3:
            mv = []
            for idx in d3.index:
                vals = [str(d3.at[idx, t + "_" + it]) for t in live3]
                if all(x in ("0", "1", "0.0", "1.0") for x in vals):
                    s = sum(int(float(x)) for x in vals)
                    mv.append(1 if s * 2 > len(live3) else 0)
            mr = 100 * sum(mv) / len(mv) if mv else None
            if mr is not None:
                maj_rates.append(mr)
            line += "  maj=" + ("NA" if mr is None else str(round(mr, 1)))
        print(line)
    pooled = {t: [] for t in live3}
    for it in ITEMS:
        for idx in d3.index:
            vals = {t: str(d3.at[idx, t + "_" + it]) for t in live3}
            if all(vals[t] in ("0", "1", "0.0", "1.0") for t in live3):
                for t in live3:
                    pooled[t].append(int(float(vals[t])))
    n_pool = len(pooled[live3[0]])
    print("item-decisions coded by all live models:", n_pool)
    if n_pool and len(live3) >= 2:
        for a, b in itertools.combinations(live3, 2):
            print("  Cohen kappa", a, "vs", b, ":",
                  round(cohen_kappa_score(pooled[a], pooled[b]), 3))
        if HAVE_SM and len(live3) >= 3:
            rows = list(zip(*[pooled[t] for t in live3]))
            tbl, _ = aggregate_raters(rows)
            print("  Fleiss kappa (" + ", ".join(live3) + ", 13-item pooled):",
                  round(fleiss_kappa(tbl), 3))
    if len(live3) >= 3 and len(maj_rates) == 13:
        print("overall mean compliance (majority vote, 13 items):",
              round(sum(maj_rates) / 13, 1), "%")

## Step 10 - How this maps to the paper

The numbers above come from **LLM annotators**, so the paper must describe them as such:
name the models and versions, the run date, and the prompt.

* **Task 2** gives each model's Tier E false-positive rate and ceiling, the pairwise and
  (with three models) Fleiss kappa, and a majority-vote ceiling.
* **Task 3** gives the pairwise and Fleiss kappa for the 13-item rubric and the per-item
  rates, per model and (with three models) by majority vote.

A third model does not make the annotators agree more; what it buys you is a majority
vote, a more stable consensus than any single model, and a genuine multi-rater Fleiss
kappa. Report the kappa honestly: if it is low, that is itself a finding about
LLM-as-annotator, not a clean human-style reliability figure.

Send the Step 9 output to have the Methodology, the Table 7 caption, and the Tier E and
compliance paragraphs rewritten to match, and the placeholders filled, only from a
complete run.